# Manual 0.2.0 dual T4 acceptance

Run manually in a fresh Kaggle session with exactly two Tesla T4 GPUs and Internet enabled. This notebook installs the unchanged PyPI 0.2.0 SDK and uses its pinned historical Hugging Face wheel. No Kaggle Input dataset is needed.

In [ ]:
PYPI_PATH = "/pypi/kaggle-vllm/0.2.0/json"
EXPECTED_GPUS = ["Tesla T4, 7.5"] * 2

import json, os, pathlib, shutil, socket, subprocess, sys, time, urllib.request
from urllib.error import HTTPError, URLError
EVIDENCE = pathlib.Path("/kaggle/working/kaggle-vllm-public-acceptance.json")
assert sys.version_info[:2] == (3, 12), sys.version
def network(host, path):
    try:
        socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
    except socket.gaierror as error:
        raise RuntimeError(f"DNS failure for {host}; enable Internet in Kaggle Notebook settings: {error}") from error
    try:
        with urllib.request.urlopen(f"https://{host}{path}", timeout=12) as response:
            return response.status
    except (HTTPError, URLError, TimeoutError) as error:
        raise RuntimeError(f"Network/PyPI/HF failure at {host}; check Internet, proxy, pip index and TLS: {error}") from error
network("pypi.org", PYPI_PATH)
network("huggingface.co", "/api/models/waqasm86/kaggle-vllm-binaries")
gpus = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], text=True, timeout=15).splitlines()
assert [x.strip() for x in gpus] == EXPECTED_GPUS, gpus
print({"python": sys.version.split()[0], "gpus": gpus, "internet": "PASS"})


In [ ]:

import shlex
subprocess.run([sys.executable, "-m", "pip", "install", "--retries", "3", "--timeout", "20", "kaggle-vllm[hub]==0.2.0"], check=True, timeout=600)
import importlib.metadata
assert importlib.metadata.version("kaggle-vllm") == "0.2.0"
subprocess.run(["kaggle-vllm", "bootstrap", "--strict"], check=True, timeout=1800)
doctor = subprocess.run(["kaggle-vllm", "doctor", "--strict", "--json"], check=True, capture_output=True, text=True, timeout=120)
doctor_data = json.loads(doctor.stdout)
assert doctor_data["compatible"] is True
print("SDK, historical wheel checksum and strict native diagnostics: PASS")


In [ ]:

from huggingface_hub import snapshot_download
model = snapshot_download(repo_id="facebook/opt-125m", revision="27dcfa74d334bc871f3234de431e71c6eeba5dd6")
before = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
script = f"""from kaggle_vllm import KaggleLLM, KaggleSamplingParams
llm=KaggleLLM({model!r}, profile="kaggle-t4x2-cu128", tensor_parallel_size=2, max_model_len=512, gpu_memory_utilization=0.40)
out=llm.generate(["NCCL enables"], KaggleSamplingParams(temperature=0.0, max_tokens=24))
assert out and out[0].outputs and out[0].outputs[0].text.strip()
print(out[0].outputs[0].text[:200])
"""
command = 'eval "$(kaggle-vllm env)"; exec ' + shlex.quote(sys.executable) + ' -c ' + shlex.quote(script)
result = subprocess.run(["bash", "-lc", command], capture_output=True, text=True, timeout=900)
assert result.returncode == 0, result.stderr[-3000:]
time.sleep(10)
after = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
assert all(int(a) <= int(b) + 1024 for a, b in zip(after.strip().splitlines(), before.strip().splitlines())), "GPU memory was not reclaimed"
evidence = {"release": "0.2.0", "status": "PASS", "gpu_profile": "T4x2/SM75", "tensor_parallel": 2,
            "native_sha256": "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c",
            "hf_revision": "f6b4f10de54924ed6fe9e28cceab84eca7276ab6",
            "generation_excerpt": result.stdout[-200:], "gpu_memory_before": before.strip().splitlines(),
            "gpu_memory_after": after.strip().splitlines(), "doctor": doctor_data}
assert len(json.dumps(evidence)) < 100_000
EVIDENCE.write_text(json.dumps(evidence, indent=2) + "\n")
print(EVIDENCE)
